# Solrのベクトル検索のシンプルなサンプル

Solrをベクトルデータベースとして使用するサンプル。

※Solrのバージョン：9.8.1

Solr 9.0以降でベクトル検索機能が追加され、DenseVectorFieldを使用してベクトル検索が可能になりました。

## 必要パッケージのインポート

In [ ]:
import pysolr
import requests
from sentence_transformers import SentenceTransformer
import time
import json

## 設定

In [ ]:
SOLR_URL = 'http://llm-rag-examples-solr:8983/solr'
CORE_NAME = 'vector_test01'  # スタンドアロンモードではコアを使用

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer(MODEL_NAME)

## Solrクライアント接続

In [ ]:
solr = pysolr.Solr(f'{SOLR_URL}/{CORE_NAME}', always_commit=True)

## コレクションの有無チェック、なければエラーで終了

In [ ]:
try:
    response = requests.get(f'{SOLR_URL}/admin/cores?action=STATUS&wt=json')
    cores_data = response.json()
    cores = list(cores_data.get('status', {}).keys())
    
    if CORE_NAME not in cores:
        raise Exception(f'Core {CORE_NAME} がありません。まず、登録処理を実行してください。（solr-vector-ex01-01-insert.ipynb）')
    else:
        print(f'Core {CORE_NAME} が存在します。')
except Exception as e:
    raise Exception(f'Error: {e}')

## ベクトル検索

In [ ]:
QUERY_TEXT = '日本の都市'

In [ ]:
query_vector = model.encode(QUERY_TEXT)
print(f"Query: {QUERY_TEXT}")
print(f"Query vector dimension: {len(query_vector)}")

### kNN検索の実行（POST方式）

In [ ]:
# Solr kNN検索（POST方式を使用）
search_url = f'{SOLR_URL}/{CORE_NAME}/select'
vector_str = '[' + ','.join(map(str, query_vector.tolist())) + ']'

search_data = {
    'q': f'{{!knn f=vector topK=3}}{vector_str}',
    'fl': 'id,text,score',
    'rows': 3,
    'wt': 'json'
}

print(f"Search query: {search_data['q'][:100]}...")

In [ ]:
# POST方式で検索実行
search_response = requests.post(
    search_url,
    data=search_data,
    headers={'Content-Type': 'application/x-www-form-urlencoded'}
)

if search_response.status_code == 200:
    search_results = search_response.json()
    print(f"Found {search_results['response']['numFound']} documents")
else:
    print(f"Search failed: {search_response.status_code}")
    print(f"Response: {search_response.text}")

In [ ]:
# 結果詳細表示
if 'response' in search_results:
    docs = search_results['response']['docs']
    for i, doc in enumerate(docs):
        print(f"Result {i+1}:")
        print(f"  ID: {doc.get('id', 'N/A')}")
        print(f"  Score: {doc.get('score', 'N/A')}")
        print(f"  Text: {doc.get('text', 'N/A')}")
        print()
else:
    print("No results found or search failed")
    print(search_results)

## 検索結果の簡潔表示

In [ ]:
print(f"クエリ: {QUERY_TEXT}")
print("=" * 50)
if 'response' in search_results:
    for doc in search_results['response']['docs']:
        score = doc.get('score', 0)
        text = doc.get('text', 'N/A')
        print(f"スコア: {score:.4f} テキスト: {text}")
else:
    print("検索結果なし")

## 別のクエリでの検索例

In [ ]:
# 別のクエリで検索
QUERY_TEXT2 = 'ペット'
query_vector2 = model.encode(QUERY_TEXT2)
vector_str2 = '[' + ','.join(map(str, query_vector2.tolist())) + ']'

search_data2 = {
    'q': f'{{!knn f=vector topK=3}}{vector_str2}',
    'fl': 'id,text,score',
    'rows': 3,
    'wt': 'json'
}

search_response2 = requests.post(
    search_url,
    data=search_data2,
    headers={'Content-Type': 'application/x-www-form-urlencoded'}
)

if search_response2.status_code == 200:
    search_results2 = search_response2.json()
    print(f"クエリ: {QUERY_TEXT2}")
    print("=" * 50)
    if 'response' in search_results2:
        for doc in search_results2['response']['docs']:
            score = doc.get('score', 0)
            text = doc.get('text', 'N/A')
            print(f"スコア: {score:.4f} テキスト: {text}")
    else:
        print("検索結果なし")
else:
    print(f"Search failed: {search_response2.status_code}")